In [ ]:
# ============================================================
# Smart Container Risk Detection Engine
# Hybrid Ensemble: XGBoost + Random Forest + Isolation Forest
# ============================================================

# --- 1. Install Dependencies ---
# !pip install pandas numpy scikit-learn xgboost openpyxl -q

In [ ]:
# --- 2. Import Libraries ---

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
)
from xgboost import XGBClassifier

In [ ]:
# --- 3. Load Datasets ---

COL_RENAME = {
    "Declaration_Date (YYYY-MM-DD)": "Declaration_Date",
    "Trade_Regime (Import / Export / Transit)": "Trade_Regime",
}


def load_data(path: str) -> pd.DataFrame:
    """Load CSV and standardize column names."""
    return pd.read_csv(path).rename(columns=COL_RENAME)


df_hist = load_data("Historical Data.csv")
df_rt = load_data("Real-Time Data.csv")

In [ ]:
# --- 4. Preprocessing & Label Creation ---

RISK_STATUSES = {"critical", "hold", "inspect", "detained", "flagged"}
NUM_COLS = ["Declared_Value", "Declared_Weight", "Measured_Weight", "Dwell_Time_Hours"]
CAT_COLS = [
    "Trade_Regime", "Origin_Country", "Destination_Country",
    "Destination_Port", "Shipping_Line",
]


def preprocess(df: pd.DataFrame, has_label: bool = True) -> pd.DataFrame:
    """Clean types, impute missing values, and derive binary risk label."""
    df = df.copy()

    # Temporal features
    df["Declaration_Date"] = pd.to_datetime(df["Declaration_Date"], errors="coerce")
    df["declaration_month"] = df["Declaration_Date"].dt.month.fillna(1).astype(int)
    df["declaration_dow"] = df["Declaration_Date"].dt.dayofweek.fillna(0).astype(int)

    # Numeric cleaning & imputation
    for col in NUM_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col].fillna(df[col].median(), inplace=True)
    for col in ["Declared_Value", "Declared_Weight"]:
        df[col] = df[col].clip(upper=df[col].quantile(0.99))

    # Categorical imputation
    for col in CAT_COLS:
        df[col].fillna(df[col].mode()[0], inplace=True)

    # Binary risk label from Clearance_Status
    if has_label and "Clearance_Status" in df.columns:
        df["Risk_Label"] = (
            df["Clearance_Status"].str.strip().str.lower()
            .isin(RISK_STATUSES).astype(int)
        )

    return df


df_hist = preprocess(df_hist, has_label=True)
df_rt = preprocess(df_rt, has_label=False)

In [ ]:
# --- 5. Feature Engineering (Leak-Free) ---
#
# CRITICAL: All statistics are computed from TRAINING data only.
# Entity risk rates use Laplace smoothing to prevent extreme values
# from rare entities, which prevents overfitting.

EPS = 1e-5
LAPLACE_ALPHA = 10  # smoothing strength for entity risk rates


def compute_train_stats(df: pd.DataFrame) -> dict:
    """Compute benchmark statistics from training data only."""
    stats = {}

    # Commodity (HS_Code) weight benchmarks
    hs_grp = df.groupby("HS_Code")["Declared_Weight"]
    stats["hs_avg"] = hs_grp.mean()
    stats["hs_std"] = hs_grp.std().fillna(1)
    stats["hs_global_avg"] = df["Declared_Weight"].mean()
    stats["hs_global_std"] = max(df["Declared_Weight"].std(), 1)

    # Exporter weight benchmarks
    exp_grp = df.groupby("Exporter_ID")["Declared_Weight"]
    stats["exp_avg"] = exp_grp.mean()
    stats["exp_std"] = exp_grp.std().fillna(1)
    stats["exp_global_avg"] = df["Declared_Weight"].mean()
    stats["exp_global_std"] = max(df["Declared_Weight"].std(), 1)

    # Dwell time benchmarks
    stats["dwell_mean"] = df["Dwell_Time_Hours"].mean()
    stats["dwell_std"] = max(df["Dwell_Time_Hours"].std(), 1)

    # Exporter / Importer / Route risk rates with Laplace smoothing
    # Formula: (sum_risk + alpha * global_rate) / (count + alpha)
    # This prevents rare entities from getting extreme 0.0 or 1.0 rates
    if "Risk_Label" in df.columns:
        global_rate = df["Risk_Label"].mean()
        stats["global_risk_rate"] = global_rate

        for col, key in [("Exporter_ID", "exporter_risk_rate"),
                         ("Importer_ID", "importer_risk_rate")]:
            grp = df.groupby(col)["Risk_Label"]
            s = grp.sum()
            c = grp.count()
            stats[key] = (s + LAPLACE_ALPHA * global_rate) / (c + LAPLACE_ALPHA)

        df = df.copy()
        df["_route"] = df["Origin_Country"] + "_" + df["Destination_Country"]
        rgrp = df.groupby("_route")["Risk_Label"]
        rs = rgrp.sum()
        rc = rgrp.count()
        stats["route_risk_score"] = (rs + LAPLACE_ALPHA * global_rate) / (rc + LAPLACE_ALPHA)

    return stats


def engineer_features(df: pd.DataFrame, train_stats: dict) -> pd.DataFrame:
    """Build all domain-specific features using pre-computed training statistics."""
    df = df.copy()

    # --- Weight difference features ---
    df["weight_diff"] = df["Measured_Weight"] - df["Declared_Weight"]
    df["weight_ratio"] = df["Measured_Weight"] / (df["Declared_Weight"] + EPS)
    df["weight_deviation_pct"] = (
        df["weight_diff"].abs() / (df["Declared_Weight"] + EPS)
    ).clip(upper=0.5)

    # --- Commodity weight benchmark ---
    df["commodity_avg_weight"] = (
        df["HS_Code"].map(train_stats["hs_avg"]).fillna(train_stats["hs_global_avg"])
    )
    df["commodity_std_weight"] = (
        df["HS_Code"].map(train_stats["hs_std"]).fillna(train_stats["hs_global_std"])
    )
    df["commodity_weight_zscore"] = (
        (df["Declared_Weight"] - df["commodity_avg_weight"])
        / (df["commodity_std_weight"] + EPS)
    ).clip(-3, 3)

    # --- Exporter weight deviation ---
    df["exporter_avg_weight"] = (
        df["Exporter_ID"].map(train_stats["exp_avg"]).fillna(train_stats["exp_global_avg"])
    )
    df["exporter_weight_deviation"] = (
        (df["Declared_Weight"] - df["exporter_avg_weight"])
        / (df["Exporter_ID"].map(train_stats["exp_std"]).fillna(train_stats["exp_global_std"]) + EPS)
    ).clip(-3, 3)

    # --- Cargo density ---
    df["density"] = df["Measured_Weight"] / (df["Declared_Value"] + EPS)

    # --- Value per weight ---
    df["value_per_weight"] = df["Declared_Value"] / (df["Declared_Weight"] + EPS)

    # --- Dwell time z-score ---
    df["dwell_time_zscore"] = (
        (df["Dwell_Time_Hours"] - train_stats["dwell_mean"])
        / (train_stats["dwell_std"] + EPS)
    ).clip(-3, 3)

    # --- Exporter / Importer / Route risk history (smoothed) ---
    global_rate = train_stats.get("global_risk_rate", 0.01)
    if "exporter_risk_rate" in train_stats:
        df["exporter_risk_rate"] = (
            df["Exporter_ID"].map(train_stats["exporter_risk_rate"]).fillna(global_rate)
        )
    else:
        df["exporter_risk_rate"] = global_rate

    if "importer_risk_rate" in train_stats:
        df["importer_risk_rate"] = (
            df["Importer_ID"].map(train_stats["importer_risk_rate"]).fillna(global_rate)
        )
    else:
        df["importer_risk_rate"] = global_rate

    if "route_risk_score" in train_stats:
        df["_route"] = df["Origin_Country"] + "_" + df["Destination_Country"]
        df["route_risk_score"] = (
            df["_route"].map(train_stats["route_risk_score"]).fillna(global_rate)
        )
        df.drop(columns=["_route"], inplace=True)
    else:
        df["route_risk_score"] = global_rate

    return df

In [ ]:
# --- 6. Encode, Split FIRST, Then Engineer (Leak-Free Pipeline) ---
#
# The key anti-overfit strategy:
#   1. Split raw data into train / val BEFORE any statistics
#   2. Compute all benchmark stats from TRAIN split only
#   3. Apply those stats to both train and val
# This prevents validation data from leaking into training features.

FEATURE_COLS = [
    # Categorical
    "Trade_Regime", "Origin_Country", "Destination_Country",
    "Destination_Port", "Shipping_Line",
    # Numeric base
    "Declared_Value", "Declared_Weight", "Measured_Weight",
    "Dwell_Time_Hours", "declaration_month", "declaration_dow",
    # Weight anomaly features
    "weight_diff", "weight_ratio", "weight_deviation_pct",
    # Commodity benchmark
    "commodity_weight_zscore",
    # Exporter weight deviation
    "exporter_weight_deviation",
    # Cargo density & value
    "density", "value_per_weight",
    # Dwell time
    "dwell_time_zscore",
    # Entity risk history (Laplace smoothed)
    "exporter_risk_rate", "importer_risk_rate", "route_risk_score",
]


def encode_categories(df, fit=True, encoders=None):
    """Label-encode categorical columns."""
    df = df.copy()
    if encoders is None:
        encoders = {}
    for col in CAT_COLS:
        if col not in df.columns:
            continue
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
        else:
            le = encoders[col]
            known = set(le.classes_)
            df[col] = df[col].astype(str).apply(
                lambda v: v if v in known else le.classes_[0]
            )
            df[col] = le.transform(df[col])
    return df, encoders


def prepare_features(df):
    """Select feature columns and target."""
    avail = [c for c in FEATURE_COLS if c in df.columns]
    X = df[avail]
    y = df["Risk_Label"].astype(int) if "Risk_Label" in df.columns else None
    return X, y


# Step 1: Stratified split on RAW preprocessed data (before any feature engineering)
indices = df_hist.index.values
y_raw = df_hist["Risk_Label"].values
tr_idx, val_idx = train_test_split(
    indices, test_size=0.2, stratify=y_raw, random_state=42
)
df_train_raw = df_hist.loc[tr_idx].copy()
df_val_raw = df_hist.loc[val_idx].copy()

# Step 2: Compute stats from TRAINING split only
train_stats = compute_train_stats(df_train_raw)

# Step 3: Engineer features using train-only stats
df_train_eng = engineer_features(df_train_raw, train_stats)
df_val_eng = engineer_features(df_val_raw, train_stats)

# Step 4: Fit encoders on training split only
df_train_enc, encoders = encode_categories(df_train_eng, fit=True)
df_val_enc, _ = encode_categories(df_val_eng, fit=False, encoders=encoders)

# Step 5: Extract feature matrices
X_train, y_train = prepare_features(df_train_enc)
X_val, y_val = prepare_features(df_val_enc)

# Step 6: Standardize numeric features
scaler = StandardScaler()
num_feats = [c for c in X_train.columns if c not in CAT_COLS]
X_train[num_feats] = scaler.fit_transform(X_train[num_feats])
X_val[num_feats] = scaler.transform(X_val[num_feats])

In [ ]:
# --- 7. Model Training (Regularized) ---

# Class imbalance weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
spw = neg_count / max(pos_count, 1)

# XGBoost — regularized but not overly constrained
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.6,
    colsample_bytree=0.5,
    min_child_weight=50,
    gamma=3.0,
    reg_alpha=5.0,
    reg_lambda=10.0,
    scale_pos_weight=spw,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)

# Random Forest — controlled complexity
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=50,
    min_samples_split=100,
    max_features=0.3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

# Isolation Forest — unsupervised anomaly detection
IF_FEATURES = [
    "weight_diff", "weight_ratio", "density",
    "commodity_weight_zscore", "dwell_time_zscore", "value_per_weight",
]

iso_model = IsolationForest(
    n_estimators=100,
    max_samples=0.5,
    contamination=0.02,
    random_state=42,
    n_jobs=-1,
)
iso_model.fit(X_train[IF_FEATURES])

# Cache Isolation Forest baseline from training data
train_if_raw = -iso_model.decision_function(X_train[IF_FEATURES])

In [ ]:
# --- 8. Ensemble Prediction & Risk Scoring ---

W_XGB, W_RF, W_IF = 0.50, 0.30, 0.20


def normalize_isolation_scores(model, X_if, baseline_raw=None):
    """Convert Isolation Forest decision function to 0-1 anomaly probability."""
    raw = -model.decision_function(X_if)
    if baseline_raw is not None:
        lo, hi = baseline_raw.min(), baseline_raw.max()
    else:
        lo, hi = raw.min(), raw.max()
    return np.clip((raw - lo) / (hi - lo + 1e-9), 0, 1)


def compute_ensemble_score(xgb, rf, iso, X, X_if, baseline_raw=None):
    """Weighted ensemble of all three models, normalized to 0-100."""
    xgb_prob = xgb.predict_proba(X)[:, 1]
    rf_prob = rf.predict_proba(X)[:, 1]
    if_score = normalize_isolation_scores(iso, X_if, baseline_raw)
    combined = W_XGB * xgb_prob + W_RF * rf_prob + W_IF * if_score
    return np.round(combined * 100, 2)


def assign_risk_label(score):
    """Convert numeric risk score to categorical label."""
    if score >= 70:
        return "High Risk"
    if score >= 30:
        return "Medium Risk"
    return "Low Risk"


# Score train & validation sets
train_scores = compute_ensemble_score(
    xgb_model, rf_model, iso_model, X_train, X_train[IF_FEATURES], train_if_raw
)
val_scores = compute_ensemble_score(
    xgb_model, rf_model, iso_model, X_val, X_val[IF_FEATURES], train_if_raw
)

In [ ]:
# --- 8b. Model Evaluation — Confusion Matrix & All Metrics ---
#
# Threshold is set to 30 (matching the Low/Medium boundary) because
# the positive class is rare (~1%) and a higher threshold would
# miss most risky containers.

THRESHOLD = 30


def evaluate_set(tag, y_true, scores):
    """Compute and display confusion matrix + all classification metrics."""
    y_pred = (scores >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    f1_w = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    auc = roc_auc_score(y_true, scores / 100)
    print(f"\n{'=' * 55}")
    print(f"  {tag} Set Evaluation  (threshold={THRESHOLD})")
    print(f"{'=' * 55}")
    print(f"  Confusion Matrix:")
    print(f"    TN={cm[0][0]:>6}   FP={cm[0][1]:>6}")
    print(f"    FN={cm[1][0]:>6}   TP={cm[1][1]:>6}")
    print(f"\n  Accuracy:          {acc:.4f}")
    print(f"  Precision:         {prec:.4f}")
    print(f"  Recall:            {rec:.4f}")
    print(f"  F1-Score:          {f1:.4f}")
    print(f"  F1-Score (wt):     {f1_w:.4f}")
    print(f"  AUC-ROC:           {auc:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_true, y_pred,
          target_names=["Safe", "Risky"], zero_division=0))
    return {"accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "f1_weighted": f1_w, "auc_roc": auc}


train_metrics = evaluate_set("Train", y_train, train_scores)
val_metrics = evaluate_set("Validation", y_val, val_scores)

# Overfit gap
print(f"\n{'=' * 55}")
print(f"  Overfit Gap (Train - Validation)")
print(f"{'=' * 55}")
for key in train_metrics:
    gap = train_metrics[key] - val_metrics[key]
    print(f"  {key:>15s}: {gap:+.4f}")

# 5-Fold Stratified Cross-Validation AUC (XGBoost)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"\n  5-Fold CV AUC (XGB): {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

# Risk label distribution
val_labels = pd.Series([assign_risk_label(s) for s in val_scores])
print(f"\n  Validation Risk Distribution: {val_labels.value_counts().to_dict()}")

In [ ]:
# --- 9. Explanation Generation ---

FEAT_DESC = {
    "weight_diff":              "High weight deviation between declared and measured",
    "weight_ratio":             "Abnormal measured-to-declared weight ratio",
    "weight_deviation_pct":     "Significant weight deviation detected",
    "commodity_weight_zscore":   "Weight unusual for this commodity type",
    "exporter_weight_deviation": "Exporter historical weight anomaly pattern detected",
    "density":                  "Abnormal cargo density detected",
    "value_per_weight":         "Unusual value-to-weight ratio",
    "dwell_time_zscore":        "Abnormal dwell time in port",
    "Dwell_Time_Hours":         "Extended port dwell time",
    "exporter_risk_rate":       "Exporter historical anomaly pattern detected",
    "importer_risk_rate":       "Importer historical risk pattern detected",
    "route_risk_score":         "High-risk trade route detected",
    "Declared_Value":           "Unusual declared cargo value",
}


def generate_explanations(model, X: pd.DataFrame, scores: np.ndarray) -> list:
    """Feature-importance-based human-readable explanations."""
    importances = model.feature_importances_
    feat_names = X.columns.tolist()
    top_indices = np.argsort(importances)[::-1][:6]
    top_feats = [feat_names[i] for i in top_indices]

    explanations = []
    for idx in range(len(X)):
        score = scores[idx]
        if score < 30:
            explanations.append("No significant risk signals detected.")
            continue

        row = X.iloc[idx]
        signals = []
        for feat in top_feats:
            desc = FEAT_DESC.get(feat, feat.replace("_", " ").title())
            val = row[feat]
            if feat in ("commodity_weight_zscore", "dwell_time_zscore",
                        "exporter_weight_deviation"):
                if abs(val) > 1.0:
                    signals.append(desc)
            elif feat == "weight_deviation_pct":
                if val > 0.08:
                    signals.append(desc)
            elif feat in ("exporter_risk_rate", "importer_risk_rate",
                          "route_risk_score"):
                if val > 0.05:
                    signals.append(desc)
            elif feat == "density":
                if abs(val) > 1.5:
                    signals.append(desc)
            else:
                signals.append(desc)
            if len(signals) >= 3:
                break

        if not signals:
            explanations.append(
                "Ensemble model detected elevated risk pattern."
                if score >= 70
                else "Moderate risk signals detected by ensemble model."
            )
        else:
            explanations.append("; ".join(signals) + ".")

    return explanations

In [ ]:
# --- 10. Real-Time Inference & Output DataFrame ---


def run_inference(df_new: pd.DataFrame) -> pd.DataFrame:
    """Full inference pipeline: engineer -> encode -> scale -> score -> explain."""
    df_eng = engineer_features(df_new, train_stats)
    df_enc, _ = encode_categories(df_eng, fit=False, encoders=encoders)
    X_new, _ = prepare_features(df_enc)
    X_new[num_feats] = scaler.transform(X_new[num_feats])

    scores = compute_ensemble_score(
        xgb_model, rf_model, iso_model,
        X_new, X_new[IF_FEATURES], train_if_raw
    )
    return pd.DataFrame({
        "container_id": df_new["Container_ID"].values,
        "risk_score": scores,
        "risk_label": [assign_risk_label(s) for s in scores],
        "explanation": generate_explanations(xgb_model, X_new, scores),
    })


# Historical predictions
results_hist = run_inference(df_hist)
results_hist.to_csv("risk_predictions.csv", index=False)

# Real-time predictions
results_rt = run_inference(df_rt)
results_rt.to_csv("risk_predictions_realtime.csv", index=False)

results_rt.head(10)